# LLM Benchmark Framework Guide

This notebook provides a step-by-step guide for setting up and running benchmark tests using the LLM Benchmark Framework.

## Step 1: Import Required Modules

First, let's import all the necessary modules for running benchmarks.

In [1]:
import asyncio
from models.ollama_model import OllamaModel, OllamaConfig
from metrics.relevance import RelevanceMetric
from metrics.evaluator import EvaluatorFactory
from benchmark.runner import BenchmarkRunner
from visualizations.evaluation_visualizer import EvaluationVisualizer

/home/benedikt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Step 2: Set Up the Model to Evaluate

Configure the model you want to evaluate. Here we'll use a local Ollama model.

In [2]:
# Configure the model to evaluate
model_config = OllamaConfig(
    model_name="llama3.2:latest",
    temperature=0.7,
    num_ctx=2048
)
model = OllamaModel(config=model_config)

## Step 3: Set Up the Evaluator

Configure the model that will evaluate the responses. This should be a model you trust to provide good evaluations.

In [3]:
# Configure the evaluator model
llama_config = OllamaConfig(
    model_name="llama3.2:latest",
    temperature=0.3,
    num_ctx=2048
)

gemma_config = OllamaConfig(
    model_name="gemma3:1b",
    temperature=0.3,
    num_ctx=2048
)
llama = OllamaModel(config=llama_config)
gemma = OllamaModel(config=gemma_config)
evaluator = EvaluatorFactory.create_evaluator([llama, gemma])

## Step 4: Define Evaluation Metrics

Choose which metrics you want to use for evaluation. You can use multiple metrics or create custom ones.

In [12]:
# Initialize metrics
metrics = [
    RelevanceMetric(),
]

## Step 5: Create the Benchmark Runner

Initialize the benchmark runner with your evaluator and metrics.

In [5]:
runner = BenchmarkRunner(evaluator, metrics)

## Step 6: Define Test Prompts

Create a list of prompts to test your model. These should cover different aspects you want to evaluate.

In [6]:
prompts = [
    "What is the capital of France?",
    "Explain the concept of quantum computing in simple terms.",
    "Write a short poem about artificial intelligence.",
    "If a train travels at 60 mph for 2.5 hours, how far does it go?",
    "Write a Python function to calculate the Fibonacci sequence."
]

## Step 7: Run the Benchmark

Execute the benchmark and get the results.

In [7]:
async def run_benchmark():
    print("Running benchmark...")
    res = await runner.run_benchmark(model, prompts)
    return res

benchmark_result = await run_benchmark()

Running benchmark...


In [ ]:
print(benchmark_result)

## Step 8: Visualize Results

Create visualizations of the benchmark results.

In [13]:
from visualizations.evaluation_visualizer import EvaluationVisualizer

# Initialize visualizer
visualizer = EvaluationVisualizer(results_dir="results")

# Plot overall results
figure = visualizer.plot_benchmark_results(benchmark_result, "Model Benchmark Results")
visualizer.save_plot(figure, "benchmark_results.png")

# Plot per-question scores
figure = visualizer.plot_per_question_scores(benchmark_result)
visualizer.save_plot(figure, "per_question_scores.png")

# Plot model summary
figure = visualizer.plot_model_summary(benchmark_result)
visualizer.save_plot(figure, "model_summary.png")

# Plot detailed results for each metric
for metric in metrics:
    figure = visualizer.plot_metric_details(benchmark_result, metric.name)
    visualizer.save_plot(figure, f"{metric.name}_details.png")

## Step 9: Analyze Results

You can access the benchmark results programmatically for further analysis.

In [ ]:
avg_scores = benchmark_result.get_average_scores_by_metric()
print("\nAverage scores by metric:")
for metric, score in avg_scores.items():
    print(f"{metric}: {score:.2f}")
print("\nDetailed results for each prompt:")
for evaluation in benchmark_result.prompt_evaluations:
    print(f"\nPrompt: {evaluation.prompt}")
    print(f"Response: {evaluation.response[:100]}...")
    for eval_result in evaluation.evaluations:
        print(f"{eval_result.metric_name}: {eval_result.score:.2f}")

## Tips for Effective Benchmarking

1. **Choose Appropriate Prompts**:
   - Cover different types of tasks
   - Include both simple and complex prompts
   - Test edge cases and potential failure modes

2. **Select Good Evaluator Models**:
   - Use models known for good reasoning
   - Consider using multiple evaluators
   - Use lower temperature settings

3. **Define Clear Metrics**:
   - Create metrics that measure what matters
   - Include both quantitative and qualitative

4. **Analyze Results Thoroughly**:
   - Identify patterns and weaknesses
   - Contextualize each evaluation